Mathematically:

$$
\text{MSE} = \frac{1}{N} \sum_{i=1}^{N} \left( z_i^2 - 2 z_i z_{q_i} + z_{q_i}^2 \right)
$$

$$
\text{MSE} = \frac{1}{N} \left( \sum_{i=1}^{N} z_i^2 - 2 \sum_{i=1}^{N} z_i z_{q_i} + \sum_{i=1}^{N} z_{q_i}^2 \right)
$$

$$
\text{MSE} = \frac{1}{N} \left( \|\mathbf{z}\|^2 + \|\mathbf{z}_q\|^2 - 2 \mathbf{z} \cdot \mathbf{z}_q \right)
$$

In [1]:
import numpy as np
import jax
import jax.numpy as jnp

## Hyperparameters

In [2]:
SEED = 23
HEIGHT = 4
WIDTH = 4
NUM_CHANNELS = 8
LATENT_DIM = NUM_CHANNELS
NUM_CODES = 32
BATCH_SIZE = 8

## Numpy

In [3]:
# Generate tensors
rng = np.random.default_rng(SEED)
codes_np = rng.standard_normal((NUM_CODES, LATENT_DIM))
z_np = rng.standard_normal((BATCH_SIZE, HEIGHT, WIDTH, NUM_CHANNELS))
z_np_flat = z_np.reshape(BATCH_SIZE*HEIGHT*WIDTH, LATENT_DIM)

In [4]:
z_np_flat.shape

(128, 8)

In [5]:
# Numpy version: Assumes num_channels = latent_dim
distances = (
            np.sum(z_np_flat ** 2, axis=-1, keepdims=True)                 # a²
            + np.sum(codes_np.T ** 2, axis=0, keepdims=True)  # b²
            - 2 * np.matmul(z_np_flat, codes_np.T)        # -2ab
        )

In [6]:
distances.shape

(128, 32)

## JAX

In [7]:
codes_jax = jnp.asarray(codes_np)
z_jax = jnp.asarray(z_np)
z_jax_flat = jnp.asarray(z_np_flat)

In [20]:
def _calculate_dists(z, codes):
    z_norm = jnp.sum(z ** 2)
    c_norm = jnp.sum(codes ** 2, axis=-1)
    z_c_prod = z @ codes.T
    return (z_norm + c_norm - 2* z_c_prod).squeeze()

In [21]:
batch_dists = jax.vmap(_calculate_dists, in_axes=(0, None))

In [22]:
jax_dists = batch_dists(z_jax_flat, codes_jax)

In [23]:
jax_dists.shape

(128, 32)

In [24]:
distances

array([[14.09492747, 11.96922091,  7.452353  , ...,  6.53791323,
        20.62098477, 24.3868615 ],
       [22.86835652, 22.45270239,  7.56489619, ..., 16.16681576,
        20.2116904 , 25.26778688],
       [16.56600573, 13.97215197,  6.81350904, ...,  8.06221437,
        22.56880123, 11.37857272],
       ...,
       [ 5.92994441, 18.7279818 , 10.71870446, ...,  6.81134592,
        28.14846886, 26.36414704],
       [11.70067205,  5.78543484,  6.47694932, ...,  6.04498077,
         5.90685069, 23.81874849],
       [16.92151336, 15.23890837,  6.64497112, ...,  8.70922238,
        21.82289903,  9.84874775]], shape=(128, 32))

In [25]:
jax_dists

Array([[14.094927 , 11.969221 ,  7.4523525, ...,  6.537913 , 20.620985 ,
        24.386862 ],
       [22.868357 , 22.452702 ,  7.5648956, ..., 16.166815 , 20.211689 ,
        25.267786 ],
       [16.566008 , 13.972153 ,  6.813509 , ...,  8.062215 , 22.568802 ,
        11.378572 ],
       ...,
       [ 5.929945 , 18.727983 , 10.718705 , ...,  6.8113465, 28.148472 ,
        26.36415  ],
       [11.700672 ,  5.7854357,  6.4769497, ...,  6.044981 ,  5.9068503,
        23.818748 ],
       [16.921515 , 15.238909 ,  6.6449714, ...,  8.709223 , 21.822899 ,
         9.848747 ]], dtype=float32)